# Module 07: Autoregressive Models (GPT from Scratch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/07-autoregressive/notebook.ipynb)

**GPU recommended:** Yes for character-level training (~3 min GPU, ~15 min CPU).

## Overview

Autoregressive language models generate text one token at a time, conditioning each new token on all
previously generated tokens. GPT (Generative Pre-trained Transformer) is the canonical example:
it uses a **decoder-only** transformer with **causal (masked) self-attention** so that position $t$
can only attend to positions $\le t$.

In this notebook we will:
1. Build a character-level tokenizer and discuss BPE
2. Implement a mini-GPT from scratch in PyTorch (embeddings, causal attention, transformer blocks, LM head)
3. Train it on a small Shakespeare-like corpus
4. Explore decoding strategies: greedy, temperature, top-k, top-p (nucleus)
5. Visualize attention patterns
6. Implement a KV cache for efficient autoregressive generation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

plt.style.use("seaborn-v0_8-whitegrid")

---
## 1. Tokenization

Before feeding text to a model we need to convert characters (or subwords) into integer indices.
We start with the simplest scheme -- **character-level tokenization** -- and then briefly illustrate
the core idea behind **Byte Pair Encoding (BPE)**, the algorithm used by GPT-2/3/4.

### 1.1 Training corpus

We use a small inline Shakespeare-like text so the notebook is fully self-contained (no downloads required).

In [ ]:
TEXT = """\
ROMEO: O, she doth teach the torches to burn bright!
It seems she hangs upon the cheek of night
Like a rich jewel in an Ethiope's ear;
Beauty too rich for use, for earth too dear!
So shows a snowy dove trooping with crows,
As yonder lady o'er her fellows shows.
The measure done, I'll watch her place of stand,
And, touching hers, make blessed my rude hand.
Did my heart love till now? forswear it, sight!
For I ne'er saw true beauty till this night.

JULIET: O Romeo, Romeo! wherefore art thou Romeo?
Deny thy father and refuse thy name;
Or, if thou wilt not, be but sworn my love,
And I'll no longer be a Capulet.
What's in a name? that which we call a rose
By any other name would smell as sweet;
So Romeo would, were he not Romeo call'd,
Retain that dear perfection which he owes
Without that title. Romeo, doff thy name,
And for that name which is no part of thee
Take all myself.

ROMEO: I take thee at thy word:
Call me but love, and I'll be new baptized;
Henceforth I never will be Romeo.

JULIET: What man art thou that thus bescreen'd in night
So stumblest on my counsel?

ROMEO: By a name
I know not how to tell thee who I am:
My name, dear saint, is hateful to myself,
Because it is an enemy to thee;
Had I it written, I would tear the word.

JULIET: My ears have not yet drunk a hundred words
Of that tongue's utterance, yet I know the sound:
Art thou not Romeo and a Montague?

ROMEO: Neither, fair saint, if either thee dislike.

JULIET: How camest thou hither, tell me, and wherefore?
The orchard walls are high and hard to climb,
And the place death, considering who thou art,
If any of my kinsmen find thee here.

ROMEO: With love's light wings did I o'er-perch these walls;
For stony limits cannot hold love out,
And what love can do that dares love attempt;
Therefore thy kinsmen are no let to me.

JULIET: If they do see thee, they will murder thee.

ROMEO: Alack, there lies more peril in thine eye
Than twenty of their swords: look thou but sweet,
And I am proof against their enmity.

JULIET: I would not for the world they saw thee here.

ROMEO: I have night's cloak to hide me from their sight;
And but thou love me, let them find me here:
My life were better ended by their hate,
Than death prorogued, wanting of thy love.

JULIET: By whose direction found'st thou out this place?

ROMEO: By love, who first did prompt me to inquire;
He lent me counsel and I lent him eyes.
I am no pilot; yet, wert thou as far
As that vast shore wash'd with the farthest sea,
I should adventure for such merchandise.

JULIET: Thou know'st the mask of night is on my face,
Else would a maiden blush bepaint my cheek
For that which thou hast heard me speak tonight.
Fain would I dwell on form, fain, fain deny
What I have spoke: but farewell compliment!
Dost thou love me? I know thou wilt say 'Ay,'
And I will take thy word: yet if thou swear'st,
Thou mayst prove false; at lovers' perjuries
Then say, Jove laughs. O gentle Romeo,
If thou dost love, pronounce it faithfully:
Or if thou think'st I am too quickly won,
I'll frown and be perverse and say thee nay,
So thou wilt woo; but else, not for the world.
In truth, fair Montague, I am too fond,
And therefore thou mayst think my 'havior light:
But trust me, gentleman, I'll prove more true
Than those that have more cunning to be strange.
"""

print(f"Corpus length: {len(TEXT)} characters")
print(f"First 200 characters:\n{TEXT[:200]}")

### 1.2 Character-level tokenizer

The simplest tokenizer maps each unique character to an integer. The vocabulary size equals the
number of distinct characters in the corpus.

In [ ]:
class CharTokenizer:
    """Character-level tokenizer: each unique character is one token."""

    def __init__(self, text):
        chars = sorted(set(text))
        self.char_to_idx = {ch: i for i, ch in enumerate(chars)}
        self.idx_to_char = {i: ch for ch, i in self.char_to_idx.items()}
        self.vocab_size = len(chars)

    def encode(self, text):
        return [self.char_to_idx[ch] for ch in text]

    def decode(self, indices):
        return "".join(self.idx_to_char[i] for i in indices)


tokenizer = CharTokenizer(TEXT)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Characters: {''.join(tokenizer.idx_to_char[i] for i in range(tokenizer.vocab_size))}")

sample = "ROMEO:"
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)
print(f"\nEncode '{sample}' -> {encoded}")
print(f"Decode {encoded} -> '{decoded}'")

### 1.3 Byte Pair Encoding (BPE) -- concept

Real GPT models use **BPE**, which iteratively merges the most frequent adjacent pair of tokens
into a new token, building a subword vocabulary. Below is a minimal illustration of one BPE merge
step -- we will not use BPE for training (our corpus is too small), but seeing the algorithm
demystifies how `tiktoken` / `sentencepiece` work under the hood.

In [ ]:
def bpe_demo(text, num_merges=5):
    """Demonstrate BPE merge steps on a small string."""
    tokens = list(text)
    print(f"Initial tokens ({len(tokens)}): {tokens[:40]}...")

    for step in range(num_merges):
        pairs = Counter()
        for a, b in zip(tokens, tokens[1:]):
            pairs[(a, b)] += 1
        if not pairs:
            break
        best_pair = pairs.most_common(1)[0]
        merged = best_pair[0][0] + best_pair[0][1]
        count = best_pair[1]
        print(f"  Step {step + 1}: merge {best_pair[0]} -> '{merged}' (count={count})")

        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair[0]:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    print(f"Final tokens ({len(tokens)}): {tokens[:40]}...")
    return tokens

_ = bpe_demo(TEXT[:300], num_merges=8)

### 1.4 Prepare training data

We encode the full corpus into a tensor and create `(input, target)` pairs using a sliding window
of length `block_size`. The target at each position is the next character.

In [ ]:
BLOCK_SIZE = 64   # context window length
BATCH_SIZE = 32

data = torch.tensor(tokenizer.encode(TEXT), dtype=torch.long)
print(f"Encoded data shape: {data.shape}")

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train: {len(train_data)} tokens, Val: {len(val_data)} tokens")


def get_batch(split):
    """Sample a random batch of (input, target) pairs."""
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([d[i : i + BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i + 1 : i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
print(f"\nBatch x shape: {xb.shape}, y shape: {yb.shape}")
print(f"x[0]: {tokenizer.decode(xb[0].tolist()[:30])}...")
print(f"y[0]: {tokenizer.decode(yb[0].tolist()[:30])}...")

---
## 2. Building a Mini-GPT

We now build a GPT-style decoder-only transformer from scratch. The architecture follows the
original GPT / GPT-2 design:

```
Token IDs -> Token Embedding + Position Embedding
          -> N x TransformerBlock(CausalSelfAttention + FFN)
          -> LayerNorm -> Linear (LM Head) -> logits
```

### 2.1 Hyperparameters

In [ ]:
N_EMBED = 64      # embedding dimension
N_HEAD = 4        # number of attention heads
N_LAYER = 4       # number of transformer blocks
DROPOUT = 0.1
LEARNING_RATE = 3e-3
MAX_ITERS = 3000
EVAL_INTERVAL = 300
EVAL_ITERS = 50

print("Mini-GPT config:")
print(f"  Embedding dim: {N_EMBED}")
print(f"  Heads: {N_HEAD}, Head dim: {N_EMBED // N_HEAD}")
print(f"  Layers: {N_LAYER}")
print(f"  Block size: {BLOCK_SIZE}")
print(f"  Vocab size: {tokenizer.vocab_size}")

### 2.2 Causal self-attention head

Each head computes $\text{Attention}(Q, K, V) = \text{softmax}\!\bigl(\frac{QK^T}{\sqrt{d_k}} + M\bigr)V$
where $M$ is a causal mask that sets future positions to $-\infty$.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal (masked) self-attention."""

    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head

        self.qkv_proj = nn.Linear(n_embed, 3 * n_embed)
        self.out_proj = nn.Linear(n_embed, n_embed)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # causal mask -- upper triangular = True means "mask out"
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1),
        )

    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)  # (B, T, 3*C)
        q, k, v = qkv.split(C, dim=-1)

        # reshape to (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # scaled dot-product attention with causal mask
        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn.masked_fill(self.mask[:T, :T], float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v  # (B, n_head, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_drop(self.out_proj(out))

        if return_attn:
            return out, attn
        return out

print("CausalSelfAttention defined.")

### 2.3 Transformer block and full GPT model

Each block: LayerNorm -> CausalSelfAttention -> residual -> LayerNorm -> FFN -> residual.
The FFN is a simple two-layer MLP with GELU activation (following GPT-2).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embed)
        self.ffn = FeedForward(n_embed, dropout)

    def forward(self, x, return_attn=False):
        if return_attn:
            attn_out, attn_weights = self.attn(self.ln1(x), return_attn=True)
            x = x + attn_out
            x = x + self.ffn(self.ln2(x))
            return x, attn_weights
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

print("TransformerBlock defined.")

In [ ]:
class MiniGPT(nn.Module):
    """
    A minimal GPT-style language model.

    Components:
      - Token embedding table
      - Learned positional embedding table
      - Stack of TransformerBlocks
      - Final LayerNorm
      - Linear LM head (projects back to vocab size)
    """

    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embed)
        self.pos_emb = nn.Embedding(block_size, n_embed)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(n_embed, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

        # weight tying: share token embedding and LM head weights
        self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, return_attn=False):
        B, T = idx.shape
        tok = self.tok_emb(idx)                          # (B, T, n_embed)
        pos = self.pos_emb(torch.arange(T, device=idx.device))  # (T, n_embed)
        x = self.drop(tok + pos)

        attn_maps = []
        for block in self.blocks:
            if return_attn:
                x, attn_w = block(x, return_attn=True)
                attn_maps.append(attn_w)
            else:
                x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        if return_attn:
            return logits, loss, attn_maps
        return logits, loss

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = MiniGPT(
    vocab_size=tokenizer.vocab_size,
    n_embed=N_EMBED,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    block_size=BLOCK_SIZE,
    dropout=DROPOUT,
).to(device)

print(f"Model parameters: {model.count_parameters():,}")
print(model)

---
## 3. Training

We train with AdamW and track both training and validation loss. The loss is the standard
cross-entropy (next-token prediction) loss. Perplexity = $e^{\text{loss}}$.

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=EVAL_ITERS):
    """Estimate mean loss over several batches for train and val splits."""
    model.eval()
    results = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[i] = loss.item()
        results[split] = losses.mean().item()
    model.train()
    return results

# sanity check: untrained loss should be ~ -ln(1/vocab_size)
init_loss = estimate_loss(model)
expected = -np.log(1.0 / tokenizer.vocab_size)
print(f"Untrained loss: train={init_loss['train']:.3f}, val={init_loss['val']:.3f}")
print(f"Expected random loss: {expected:.3f}")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

train_losses = []
val_losses = []
log_steps = []

torch.manual_seed(42)
model.train()
t0 = time.time()

for step in range(MAX_ITERS):
    # periodically evaluate
    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss(model)
        train_losses.append(losses["train"])
        val_losses.append(losses["val"])
        log_steps.append(step)
        elapsed = time.time() - t0
        ppl = np.exp(losses["val"])
        print(
            f"step {step:5d} | train loss {losses['train']:.3f} | "
            f"val loss {losses['val']:.3f} | val ppl {ppl:.1f} | "
            f"time {elapsed:.1f}s"
        )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    # gradient clipping (standard practice for transformers)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

total_time = time.time() - t0
print(f"\nTraining complete in {total_time:.1f}s")

### 3.1 Training loss curve and perplexity

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(log_steps, train_losses, label="Train", linewidth=2)
ax1.plot(log_steps, val_losses, label="Validation", linewidth=2)
ax1.set_xlabel("Step")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training and Validation Loss")
ax1.legend()

train_ppl = [np.exp(l) for l in train_losses]
val_ppl = [np.exp(l) for l in val_losses]
ax2.plot(log_steps, train_ppl, label="Train", linewidth=2)
ax2.plot(log_steps, val_ppl, label="Validation", linewidth=2)
ax2.set_xlabel("Step")
ax2.set_ylabel("Perplexity")
ax2.set_title("Perplexity over Training")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final val loss: {val_losses[-1]:.3f}, val perplexity: {val_ppl[-1]:.1f}")

---
## 4. Decoding Strategies

Given a trained model, there are many ways to sample the next token from the predicted distribution.
We implement four common strategies:

| Strategy | Description |
|----------|-------------|
| **Greedy** | Always pick the most likely token. Deterministic but repetitive. |
| **Temperature** | Divide logits by $T$ before softmax. $T < 1$ sharpens, $T > 1$ flattens. |
| **Top-k** | Zero out all but the $k$ highest-probability tokens, then sample. |
| **Top-p (nucleus)** | Keep the smallest set of tokens whose cumulative probability $\ge p$, then sample. |

In [ ]:
def top_k_filter(logits, k):
    """Zero out all logits except the top-k."""
    if k == 0 or k >= logits.size(-1):
        return logits
    values, _ = torch.topk(logits, k)
    min_val = values[:, -1].unsqueeze(-1)
    return torch.where(logits < min_val, torch.full_like(logits, float("-inf")), logits)


def top_p_filter(logits, p):
    """Nucleus sampling: keep smallest set of tokens with cumulative prob >= p."""
    if p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

    # remove tokens with cumulative probability above the threshold
    # shift right so that the token that crosses the threshold is kept
    sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= p
    sorted_logits[sorted_mask] = float("-inf")

    # scatter back to original ordering
    original_logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)
    return original_logits


@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=0, top_p=1.0, greedy=False):
    """Autoregressive text generation with various decoding strategies."""
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # crop to block_size if context is too long
        idx_cond = idx[:, -model.block_size :]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # last time step

        if greedy:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            logits = top_k_filter(logits, top_k)
            logits = top_p_filter(logits, top_p)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_id], dim=1)

    return tokenizer.decode(idx[0].tolist())

print("Generation functions defined.")

### 4.1 Greedy decoding

In [ ]:
torch.manual_seed(42)
print("=== Greedy Decoding ===")
print(generate(model, "ROMEO:", max_new_tokens=200, greedy=True))

### 4.2 Temperature sampling

Low temperature (0.5) produces more focused/conservative text; high temperature (1.5) produces
more diverse but noisier text.

In [ ]:
for temp in [0.5, 0.8, 1.0, 1.5]:
    torch.manual_seed(42)
    print(f"=== Temperature = {temp} ===")
    print(generate(model, "ROMEO:", max_new_tokens=150, temperature=temp))
    print()

### 4.3 Top-k sampling

In [ ]:
for k in [3, 5, 10]:
    torch.manual_seed(42)
    print(f"=== Top-k = {k} ===")
    print(generate(model, "JULIET:", max_new_tokens=150, temperature=0.8, top_k=k))
    print()

### 4.4 Top-p (nucleus) sampling

In [ ]:
for p in [0.5, 0.9, 0.95]:
    torch.manual_seed(42)
    print(f"=== Top-p = {p} ===")
    print(generate(model, "JULIET:", max_new_tokens=150, temperature=0.8, top_p=p))
    print()

### 4.5 Visualizing the effect of temperature on probability distribution

Let us look at how temperature reshapes the next-token probability distribution for a single
prediction step.

In [ ]:
model.eval()
prompt_ids = torch.tensor([tokenizer.encode("ROMEO: ")], dtype=torch.long, device=device)
with torch.no_grad():
    logits, _ = model(prompt_ids)
    logits = logits[0, -1, :]  # logits for next token after "ROMEO: "

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
temperatures = [0.3, 1.0, 2.0]

for ax, temp in zip(axes, temperatures):
    probs = F.softmax(logits / temp, dim=-1).cpu().numpy()
    top_indices = np.argsort(probs)[-15:][::-1]
    top_probs = probs[top_indices]
    top_chars = [tokenizer.idx_to_char[i] for i in top_indices]
    labels = [repr(c) for c in top_chars]

    ax.barh(range(len(top_probs)), top_probs[::-1], color="steelblue")
    ax.set_yticks(range(len(top_probs)))
    ax.set_yticklabels(labels[::-1], fontsize=9)
    ax.set_xlabel("Probability")
    ax.set_title(f"T = {temp}")
    ax.set_xlim(0, max(0.5, top_probs[0] * 1.1))

plt.suptitle("Next-token distribution after 'ROMEO: ' at different temperatures", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Visualizing Attention Patterns

We can extract the attention weights from each layer/head and visualize them as heatmaps.
Causal attention patterns should show the characteristic lower-triangular structure (each position
attends only to itself and earlier positions).

In [ ]:
sample_text = "ROMEO: O, she doth teach"
sample_ids = torch.tensor([tokenizer.encode(sample_text)], dtype=torch.long, device=device)
T = sample_ids.shape[1]

model.eval()
with torch.no_grad():
    _, _, attn_maps = model(sample_ids, return_attn=True)

chars = list(sample_text)
print(f"Input: '{sample_text}' ({T} tokens)")

In [ ]:
fig, axes = plt.subplots(N_LAYER, N_HEAD, figsize=(3 * N_HEAD, 3 * N_LAYER))

for layer_idx in range(N_LAYER):
    attn = attn_maps[layer_idx][0].cpu().numpy()  # (n_head, T, T)
    for head_idx in range(N_HEAD):
        ax = axes[layer_idx, head_idx]
        im = ax.imshow(attn[head_idx], cmap="Blues", vmin=0, vmax=1, aspect="auto")
        if layer_idx == 0:
            ax.set_title(f"Head {head_idx}", fontsize=10)
        if head_idx == 0:
            ax.set_ylabel(f"Layer {layer_idx}", fontsize=10)
        ax.set_xticks(range(T))
        ax.set_yticks(range(T))
        if T <= 30:
            ax.set_xticklabels(chars, fontsize=5, rotation=90)
            ax.set_yticklabels(chars, fontsize=5)
        else:
            ax.set_xticklabels([])
            ax.set_yticklabels([])

plt.suptitle("Attention Weights per Layer and Head", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.1 Average attention distance per head

We can measure how "local" vs. "global" each head's attention is by computing the average distance
between the query position and the positions it attends to. A head that focuses on nearby tokens
has a small average distance; one that looks far back has a large distance.

In [ ]:
avg_distances = []

for layer_idx in range(N_LAYER):
    attn = attn_maps[layer_idx][0].cpu().numpy()  # (n_head, T, T)
    for head_idx in range(N_HEAD):
        w = attn[head_idx]  # (T, T) -- w[i,j] = attention from position i to j
        total_dist = 0.0
        for i in range(T):
            for j in range(i + 1):  # causal: only j <= i
                total_dist += w[i, j] * (i - j)
        avg_dist = total_dist / T
        avg_distances.append((layer_idx, head_idx, avg_dist))

fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"L{l}H{h}" for l, h, _ in avg_distances]
distances = [d for _, _, d in avg_distances]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(distances)))
ax.bar(labels, distances, color=colors)
ax.set_ylabel("Average Attention Distance")
ax.set_xlabel("Layer / Head")
ax.set_title("Average Attention Distance per Head")
plt.tight_layout()
plt.show()

---
## 6. KV Cache for Efficient Autoregressive Generation

During autoregressive generation, we generate tokens one at a time. Without a cache, we
recompute attention over the entire sequence for each new token. With a **KV cache**, we store
the key and value tensors from previous steps and only compute the new token's query, key, and
value -- then append to the cache. This avoids redundant computation and provides a significant
speedup, especially for long sequences.

Below we implement a KV-cache-enabled version of our attention and measure the speedup.

In [ ]:
class CausalSelfAttentionWithCache(nn.Module):
    """Causal self-attention with optional KV cache for fast autoregressive inference."""

    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head

        self.qkv_proj = nn.Linear(n_embed, 3 * n_embed)
        self.out_proj = nn.Linear(n_embed, n_embed)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1),
        )

    def forward(self, x, kv_cache=None):
        B, T_new, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T_new, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T_new, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T_new, self.n_head, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k], dim=2)
            v = torch.cat([v_prev, v], dim=2)

        new_cache = (k, v)
        T_full = k.shape[2]

        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        # apply causal mask only for the new query positions
        # q has shape (B, H, T_new, d), k has shape (B, H, T_full, d)
        # we need mask of shape (T_new, T_full)
        if T_new > 1:
            attn = attn.masked_fill(self.mask[:T_new, :T_full], float("-inf"))
        # when T_new == 1 (single token generation), no masking needed:
        # the single query can attend to all cached positions

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T_new, C)
        out = self.resid_drop(self.out_proj(out))
        return out, new_cache

print("CausalSelfAttentionWithCache defined.")

In [ ]:
class TransformerBlockWithCache(nn.Module):
    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttentionWithCache(n_embed, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embed)
        self.ffn = FeedForward(n_embed, dropout)

    def forward(self, x, kv_cache=None):
        attn_out, new_cache = self.attn(self.ln1(x), kv_cache=kv_cache)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache


class MiniGPTWithCache(nn.Module):
    """MiniGPT with KV cache support for efficient autoregressive generation."""

    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.n_layer = n_layer
        self.tok_emb = nn.Embedding(vocab_size, n_embed)
        self.pos_emb = nn.Embedding(block_size, n_embed)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlockWithCache(n_embed, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, past_kv=None, use_cache=False):
        B, T_new = idx.shape

        if past_kv is not None:
            # positions start after the cached tokens
            T_past = past_kv[0][0].shape[2]  # cached sequence length
            positions = torch.arange(T_past, T_past + T_new, device=idx.device)
        else:
            positions = torch.arange(T_new, device=idx.device)

        tok = self.tok_emb(idx)
        pos = self.pos_emb(positions)
        x = self.drop(tok + pos)

        new_kv = []
        for i, block in enumerate(self.blocks):
            layer_cache = past_kv[i] if past_kv is not None else None
            x, cache = block(x, kv_cache=layer_cache)
            if use_cache:
                new_kv.append(cache)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if use_cache:
            return logits, new_kv
        return logits, None

print("MiniGPTWithCache defined.")

### 6.1 Copy weights from trained model to cached model

We copy the learned weights from our trained `MiniGPT` into the `MiniGPTWithCache` architecture
so we can compare generation speed on an identical model.

In [ ]:
cached_model = MiniGPTWithCache(
    vocab_size=tokenizer.vocab_size,
    n_embed=N_EMBED,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    block_size=BLOCK_SIZE,
    dropout=DROPOUT,
).to(device)

# map weights from trained model to cached model
src = model.state_dict()
dst = cached_model.state_dict()

mapping = {
    "tok_emb.weight": "tok_emb.weight",
    "pos_emb.weight": "pos_emb.weight",
    "ln_f.weight": "ln_f.weight",
    "ln_f.bias": "ln_f.bias",
}

for i in range(N_LAYER):
    for suffix in [
        "ln1.weight", "ln1.bias", "ln2.weight", "ln2.bias",
        "attn.qkv_proj.weight", "attn.qkv_proj.bias",
        "attn.out_proj.weight", "attn.out_proj.bias",
        "ffn.net.0.weight", "ffn.net.0.bias",
        "ffn.net.2.weight", "ffn.net.2.bias",
    ]:
        mapping[f"blocks.{i}.{suffix}"] = f"blocks.{i}.{suffix}"

for dst_key, src_key in mapping.items():
    dst[dst_key] = src[src_key]

cached_model.load_state_dict(dst, strict=False)
cached_model.eval()
print("Weights copied to cached model.")

### 6.2 Generation with KV cache

In [ ]:
@torch.no_grad()
def generate_with_cache(model, prompt, max_new_tokens=200, temperature=0.8):
    """Generate text using KV cache -- only feed the latest token at each step."""
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

    # prefill: process the entire prompt at once
    logits, past_kv = model(idx, past_kv=None, use_cache=True)
    logits = logits[:, -1, :] / temperature
    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    generated = [next_id.item()]

    # decode step by step, feeding only the new token each time
    for _ in range(max_new_tokens - 1):
        logits, past_kv = model(next_id, past_kv=past_kv, use_cache=True)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        generated.append(next_id.item())

    return prompt + tokenizer.decode(generated)


# verify output looks reasonable
torch.manual_seed(42)
print("=== Generation with KV cache ===")
print(generate_with_cache(cached_model, "ROMEO:", max_new_tokens=150))

### 6.3 Benchmarking: with vs. without KV cache

We compare the wall-clock time to generate sequences of increasing length, with and without
the KV cache. The cache avoids re-computing attention for all previous tokens at each step.

In [ ]:
gen_lengths = [50, 100, 200, 400]
times_no_cache = []
times_with_cache = []
n_runs = 3  # average over multiple runs for stability

for length in gen_lengths:
    # without cache (original model)
    elapsed = 0.0
    for _ in range(n_runs):
        torch.manual_seed(42)
        t0 = time.time()
        _ = generate(model, "ROMEO:", max_new_tokens=length, temperature=0.8)
        elapsed += time.time() - t0
    times_no_cache.append(elapsed / n_runs)

    # with cache
    elapsed = 0.0
    for _ in range(n_runs):
        torch.manual_seed(42)
        t0 = time.time()
        _ = generate_with_cache(cached_model, "ROMEO:", max_new_tokens=length, temperature=0.8)
        elapsed += time.time() - t0
    times_with_cache.append(elapsed / n_runs)

for length, t_no, t_with in zip(gen_lengths, times_no_cache, times_with_cache):
    speedup = t_no / t_with if t_with > 0 else float("inf")
    print(f"Tokens: {length:4d} | No cache: {t_no:.4f}s | With cache: {t_with:.4f}s | Speedup: {speedup:.2f}x")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(gen_lengths, times_no_cache, "o-", label="Without KV cache", linewidth=2, markersize=6)
ax1.plot(gen_lengths, times_with_cache, "s-", label="With KV cache", linewidth=2, markersize=6)
ax1.set_xlabel("Number of Generated Tokens")
ax1.set_ylabel("Time (seconds)")
ax1.set_title("Generation Time: Cache vs. No Cache")
ax1.legend()

speedups = [t_no / t_with if t_with > 0 else 0 for t_no, t_with in zip(times_no_cache, times_with_cache)]
ax2.bar([str(l) for l in gen_lengths], speedups, color="steelblue")
ax2.set_xlabel("Number of Generated Tokens")
ax2.set_ylabel("Speedup (x)")
ax2.set_title("KV Cache Speedup Factor")
ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### 6.4 Why KV cache matters

Without the cache, generating $N$ tokens requires $O(N^2)$ attention computations (each of the
$N$ steps processes the full sequence up to that point). With the KV cache, each step only
processes a single new token against the cached keys/values, reducing total work to $O(N)$.

For our small model the absolute times are modest, but for production models with billions of
parameters and thousands of tokens, the KV cache is essential for practical inference speed.

---
## 7. Summary

In this notebook we built a complete mini-GPT from scratch:

1. **Tokenization**: Implemented a character-level tokenizer and demonstrated the core BPE algorithm that powers real GPT models.

2. **Model architecture**: Built token embeddings, positional embeddings, multi-head causal self-attention, transformer blocks with pre-norm residual connections, and a language model head with weight tying.

3. **Training**: Trained on a small Shakespeare corpus using AdamW with gradient clipping, tracking loss and perplexity over time.

4. **Decoding strategies**: Implemented and compared greedy, temperature, top-k, and top-p (nucleus) sampling -- the same strategies used in production LLM APIs.

5. **Attention visualization**: Extracted and plotted attention patterns across layers and heads, revealing how different heads specialize (some attend locally, others look further back).

6. **KV cache**: Implemented a key-value cache that avoids redundant computation during autoregressive generation, and measured the resulting speedup.

**Key takeaways**:
- Autoregressive models factorize $P(\text{text}) = \prod_t P(x_t | x_{<t})$ and use causal masking to enforce this left-to-right dependency.
- The same architecture scales from our tiny model to GPT-4 -- the core building blocks are identical.
- Decoding strategy has a large impact on generation quality; temperature and nucleus sampling are the most commonly used in practice.
- KV caching is a simple but critical optimization that makes autoregressive generation practical at scale.